# Namma Space — Autonomous 3D Gaussian Splatting Reconstruction Pipeline
**Techfest IIT Bombay 2026–27 | Namma Space Challenge (Round 1)**

This notebook trains your own authentic, end-to-end 3D Gaussian Splatting digital twin from your smartphone walkthrough video using an NVIDIA T4 GPU on Google Colab.

---
### ⚡ 2-Step Quick Start:
1. **GPU Check**: In the top menu, go to **Runtime → Change runtime type** → select **T4 GPU** → click **Save**.
2. **Run Step 1**: Prepares Python 3.10, patches GLM & GCC headers, and compiles the native CUDA rasterizers (~1.5 minutes).
3. **Run Step 2**: Discovers your uploaded video (`.mp4`), extracts frames, computes 3D camera poses via COLMAP, trains 3DGS (30k iterations), and automatically downloads `namma_space_assets.zip` to your laptop!

### Step 1: Automated Environment Setup & CUDA 12 Rasterizer Build
Sets up an isolated Python 3.10 environment, installs PyTorch CUDA 12.1, downloads GLM math headers, applies the GCC 12/13 `<cstdint>` header patches, and compiles the rasterizers natively.

In [ ]:
%%bash
set -e

echo "=== 1. Checking NVIDIA GPU ==="
nvidia-smi

echo "=== 2. Setting Up Isolated Python 3.10 Environment ==="
cd /content
if [ ! -f /content/bin/micromamba ]; then
    wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
fi

if [ ! -d /content/py310 ]; then
    ./bin/micromamba create -p /content/py310 python=3.10 -y -c conda-forge
fi

echo "=== 3. Installing PyTorch (CUDA 12.1) & Build Tools in Python 3.10 ==="
/content/py310/bin/pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
/content/py310/bin/pip install -q "setuptools<70.0.0" wheel ninja plyfile tqdm pycolmap typer rich opencv-python-headless scipy

echo "=== 4. Cloning Repositories & Populating GLM Math Headers ==="
cd /content
if [ ! -d "/content/roomsplat" ]; then
    git clone -q https://github.com/Ahmedn1/roomsplat.git
fi

# Clean stale clone and use Colab-optimized repository
if [ ! -d "/content/gaussian-splatting" ]; then
    git clone -q --recursive https://github.com/camenduru/gaussian-splatting.git /content/gaussian-splatting
fi

# Ensure GLM C++ geometry headers are physically present
mkdir -p /content/gaussian-splatting/submodules/diff-gaussian-rasterization/third_party
if [ ! -f "/content/gaussian-splatting/submodules/diff-gaussian-rasterization/third_party/glm/glm/glm.hpp" ]; then
    echo "Downloading GLM geometry headers..."
    rm -rf /content/gaussian-splatting/submodules/diff-gaussian-rasterization/third_party/glm
    git clone --depth 1 -q https://github.com/g-truc/glm.git /content/gaussian-splatting/submodules/diff-gaussian-rasterization/third_party/glm
fi

# Patch GCC 12/13 standard header inclusions
sed -i '1i #include <cstdint>\n#include <float.h>' /content/gaussian-splatting/submodules/diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h 2>/dev/null || true
sed -i '1i #include <float.h>' /content/gaussian-splatting/submodules/simple-knn/simple_knn.cu 2>/dev/null || true

cd /content/roomsplat
/content/py310/bin/pip install -q -e .

echo "=== 5. Compiling Inria CUDA Rasterizers Natively (~45s) ==="
export CUDA_HOME=/usr/local/cuda
export PATH=$CUDA_HOME/bin:$PATH

# Compile diff-gaussian-rasterization directly
/content/py310/bin/pip install --no-build-isolation /content/gaussian-splatting/submodules/diff-gaussian-rasterization

# Compile simple-knn directly
/content/py310/bin/pip install --no-build-isolation /content/gaussian-splatting/submodules/simple-knn

echo "=== 6. Pre-Flight Hardware & Rasterizer Verification ==="
/content/py310/bin/python -c "
import torch
assert torch.cuda.is_available(), 'CUDA not detected on PyTorch!'
import diff_gaussian_rasterization
import simple_knn
import pycolmap
print(f'\n[PASS] PyTorch {torch.__version__} with GPU: {torch.cuda.get_device_name(0)}')
print('[PASS] diff_gaussian_rasterization: Ready (Native CUDA 12)')
print('[PASS] simple_knn: Ready (Native CUDA 12)')
print('[PASS] pycolmap: Ready')
print('\n' + '='*65)
print('  SUCCESS: PIPELINE IS 100% READY FOR TRAINING!')
print('  Run Step 2 below to process your video.')
print('='*65)
"

### Step 2: 1-Click Video Reconstruction & Web Twin Download
Discovers your uploaded video (or prompts you to select one), extracts frames, solves camera poses, trains 3DGS for 30,000 iterations, and downloads `namma_space_assets.zip` to your computer.

In [ ]:
import os, sys, glob, shutil
from pathlib import Path
from google.colab import files

# 1. Auto-detect uploaded video in /content
video_candidates = [
    f for f in glob.glob("/content/*")
    if f.lower().endswith((".mp4", ".mov", ".m4v")) and not f.startswith("/content/sample_data")
]

if not video_candidates:
    print("No video found in /content. Please select your room walkthrough video to upload:")
    uploaded = files.upload()
    video_candidates = list(uploaded.keys())
    if not video_candidates:
        raise FileNotFoundError("No video provided. Please upload an MP4/MOV room walkthrough.")

video_file = Path(video_candidates[0]).resolve()
video_size_mb = video_file.stat().st_size / (1024 * 1024)
print(f"\nTarget Video: {video_file.name} ({video_size_mb:.2f} MB)")

# Define output directories
out_dir = Path(f"/content/{video_file.stem}_splat")
sparse_dir = out_dir / "sparse" / "0"
viewer_dir = out_dir / "viewer"
viewer_dir.mkdir(parents=True, exist_ok=True)

# =============================================================================
# STAGE 1: Frame Extraction & Camera Pose Estimation (COLMAP)
# =============================================================================
if not (sparse_dir / "cameras.bin").exists():
    print("\n--- STAGE 1: Extracting Frames & Solving Camera Trajectory ---")
    %cd /content/roomsplat
    !/content/py310/bin/python -m roomsplat.cli run "{video_file}" \
        --name "Namma Space Digital Twin" \
        --fps 1.5 \
        --max-dim 1600 \
        --skip-train \
        --no-view
else:
    print(f"\nReusing existing camera poses in: {sparse_dir}")

# =============================================================================
# STAGE 2: 3D Gaussian Splatting Training (30,000 Iterations)
# =============================================================================
print("\n--- STAGE 2: Training 3D Gaussian Radiance Field (30k Iterations on T4 GPU) ---")
%cd /content/gaussian-splatting

!/content/py310/bin/python train.py \
    -s "{out_dir}" \
    -m "{out_dir}/model" \
    --iterations 30000 \
    --disable_viewer \
    --densify_until_iter 8000 \
    --densification_interval 200

# =============================================================================
# STAGE 3: Convert to 32-Byte scene.splat & Export cameras.json
# =============================================================================
print("\n--- STAGE 3: Packaging Web-Optimized Binary 3D Twin ---")
export_script = f"""
import sys, shutil
from pathlib import Path
sys.path.insert(0, '/content/roomsplat')
from roomsplat.pipeline.export import ply_to_splat, write_cameras_json

out_dir = Path('{out_dir}')
sparse_dir = out_dir / 'sparse' / '0'
viewer_dir = out_dir / 'viewer'
viewer_dir.mkdir(parents=True, exist_ok=True)

ply_files = list(out_dir.glob('model/point_cloud/iteration_*/point_cloud.ply'))
ply_path = sorted(ply_files, key=lambda p: int(p.parent.name.split('_')[1]))[-1]
print(f'Found trained point cloud: {{ply_path}}')

splat_path = viewer_dir / 'scene.splat'
ply_to_splat(ply_path, splat_path)
write_cameras_json(sparse_dir, viewer_dir)

shutil.copy(splat_path, '/content/scene.splat')
shutil.copy(viewer_dir / 'cameras.json', '/content/cameras.json')
"""

Path("/content/export_bundle.py").write_text(export_script)
!/content/py310/bin/python /content/export_bundle.py

# Zip together
!zip -j /content/namma_space_assets.zip /content/scene.splat /content/cameras.json

# =============================================================================
# STAGE 4: Automated Download to Local Machine
# =============================================================================
print("\n=================================================================")
print("  SUCCESS: 3D Twin Generation Complete!")
print("  Downloading namma_space_assets.zip to your computer...")
print("=================================================================\n")
files.download("/content/namma_space_assets.zip")